# SRD Honest Benchmark — Colab Runner

Runs the SRD perplexity sweep on TinyLlama-1.1B and produces the data behind `docs/SRD_RESULTS.md`.

**Pipeline:** install → unit tests → coherence smoke test → SRD perplexity sweep → K-quant baseline → plot → download.

**Hardware:** Colab T4 (free tier). Total wall-clock ~30-45 min including model download. Most cells are fast; the SRD sweep (Cell 5) is the long pole at ~10-15 min.

**Phase C deliverables (downloaded in Cell 8):**
- `srd_sweep.json` — rows 1-5 of the results table (FP16 + 4 SRD configs)
- `kquant_sweep.json` — rows 6-9 (K-quant baselines, cited)
- `srd_perplexity_vs_bpw.png` — the headline plot

Paste the two JSON files back into the chat and the write-up at `docs/SRD_RESULTS.md` gets filled in.

**Phase D** (Mistral-7B scale-up) is in Cells D1–D4 at the bottom. Requires A100/H100.

**Source:** github.com/Orivael-Dev/axiom · plan at `/root/.claude/plans/abundant-wandering-salamander.md`

In [ ]:
# Cell 1: Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Sweep will run on CPU (~3 hours instead of ~15 min).")
    print("Runtime → Change runtime type → T4 GPU")

In [ ]:
# Cell 2: Clone the AXIOM repo + install research/quant requirements
#
# GIT_REF points at the SRD prototype branch. Change to 'main' once
# the PR is merged.
import os
GIT_REF = "claude/srd-prototype-benchmark-JRtv1"

!rm -rf axiom
!git clone --depth 1 --branch {GIT_REF} https://github.com/Orivael-Dev/axiom.git
os.chdir("axiom")
!pip install -q -r research/quant/requirements.txt
print("\n[setup] ready. branch:", GIT_REF)


In [ ]:
# Cell 3: Phase A — unit tests (no model download, ~2 sec)
#
# Confirms the kernel reproduces the same numbers the unit tests pin:
#   - bpw matches hand calc (4 + 8 + 32/G + 32/G)
#   - residue strictly improves MSE
#   - shape / dtype / value-range invariants hold
# If any of these fail, halt — the sweep below is meaningless until
# the kernel is right.
import os
os.environ.setdefault("AXIOM_MASTER_KEY", "colab_smoke_key_only")
!python -m pytest tests/test_axiom_quant.py -q

In [ ]:
# Cell 4: Phase B — coherence smoke test
#
# Downloads TinyLlama-1.1B (~2 GB cached after first run), applies
# SRD at alpha=1, generates 80 tokens. The output should be coherent
# English. If the model spits gibberish at alpha=1, the kernel is
# bug-bait — STOP HERE and re-check axiom_quant.py.
#
# At alpha=0 the output should be noticeably degraded but still
# English-shaped (4-bit only is lossier than the published
# Q4_K_M numbers because there's no per-block scale tuning).
!python research/quant/quantize_model.py \
    --model TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
    --alpha 1.0 \
    --prompt "Once upon a time, in a small village, " \
    --tokens 60

In [ ]:
# Cell 5: Phase C1 — SRD perplexity sweep (rows 1-5)
#
# Configs run: FP16 baseline, SRD alpha={0, 0.5, 1.0} per-block g=64,
# SRD alpha=1.0 per-tensor. Each config gets fresh weights so the
# previous run can't contaminate the next. Sliding-window PPL with
# stride=512, context=2048 (standard llama.cpp conventions).
#
# Sanity warning fires if the FP16 baseline drifts >0.5 from the
# published TinyLlama PPL (~7.7) — that means the eval harness is
# broken, not the kernel.
#
# Wall-clock: ~10-15 min on T4. The dataset fingerprint guard
# writes results/wikitext_fingerprint.txt on first run.
!python -m research.quant.bench_perplexity \
    --model TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
    --group-size 64 \
    --output research/quant/results/srd_sweep.json

In [ ]:
# Cell 6: Phase C2 — K-quant baseline (rows 6-9)
#
# Cites the published llama.cpp PPLs for TinyLlama-1.1B at
# Q4_K_M / Q5_K_M / Q6_K / Q8_0. Fast (~1 sec, no download).
# Stride convention may differ from our SRD eval — see write-up.
#
# For stride-matched apples-to-apples, run Cell 6A then Cell 6B
# instead of this cell. They build llama.cpp from source (cmake)
# and run llama-perplexity --ppl-stride 512. ~30 min on H100.

!python -m research.quant.bench_llamacpp \
    --model TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
    --output research/quant/results/kquant_sweep.json
print("Done — source=published_cite (stride caveat applies)")


In [ ]:
# Cell 6A: Install llama-cpp-python with CUDA — verified
# Installs from source with CUDA forced on. Takes ~5-8 min but
# guaranteed GPU execution. Pre-built wheels silently fall back to
# CPU if the CUDA version doesn't match exactly.
import subprocess, sys, os, time

os.environ['CMAKE_ARGS'] = '-DGGML_CUDA=on'
os.environ['FORCE_CMAKE'] = '1'
print('Building llama-cpp-python with CUDA (takes ~5-8 min)...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'llama-cpp-python',
     '--no-cache-dir', '--upgrade', '-q'],
    check=True
)
print('Install done.')

# Verify CUDA is actually compiled in
from llama_cpp import llama_cpp as _llama_lib
has_cuda = hasattr(_llama_lib, 'ggml_cuda_device_info') or hasattr(_llama_lib, 'llama_get_device_info')
if has_cuda:
    print('CUDA symbols found — GPU inference available.')
else:
    # Fallback check: try creating a tiny context and look for CUDA output
    print('CUDA symbols not detected in shared library.')
    print('If GPU is still not used, re-run this cell or check nvidia-smi.')

import subprocess as _sp
_sp.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'])
print('GPU visible above — if llama-cpp-python still runs slow, CUDA build failed.')


In [ ]:
# Cell 6B: K-quant PPL via llama-cpp-python
# Requires Cell 6A. Run Cell 6A fresh if this cell was slow before.
import math, json, time, numpy as np
from pathlib import Path
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

GGUF_REPO = 'TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF'
GGUF_DIR  = Path('/tmp/srd_gguf')
GGUF_DIR.mkdir(exist_ok=True)
OUTPUT    = Path('research/quant/results/kquant_sweep.json')
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
CONTEXT, STRIDE = 2048, 512

QUANTS = {
    'Q4_K_M': {'bpw': 4.85, 'file': 'tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf'},
    'Q5_K_M': {'bpw': 5.69, 'file': 'tinyllama-1.1b-chat-v1.0.Q5_K_M.gguf'},
    'Q6_K':   {'bpw': 6.56, 'file': 'tinyllama-1.1b-chat-v1.0.Q6_K.gguf'},
    'Q8_0':   {'bpw': 8.50, 'file': 'tinyllama-1.1b-chat-v1.0.Q8_0.gguf'},
}

# ── GPU speed check — MUST pass before running full benchmark ────────
# Download Q4_K_M first (needed for check)
dest_check = GGUF_DIR / QUANTS['Q4_K_M']['file']
if not dest_check.exists():
    print('Downloading Q4_K_M for speed check...')
    hf_hub_download(repo_id=GGUF_REPO, filename=QUANTS['Q4_K_M']['file'],
                    local_dir=str(GGUF_DIR))

print('Running GPU speed check (5 windows)...')
_llm = Llama(model_path=str(dest_check), n_ctx=CONTEXT,
             n_gpu_layers=-1, logits_all=True, verbose=True)
_tok = _llm.tokenize(b'The quick brown fox jumps over the lazy dog ' * 200, add_bos=True)
_t0 = time.monotonic()
for _ in range(5):
    _llm.reset()
    _llm.eval(_tok[:CONTEXT])
_speed = (5 * CONTEXT) / (time.monotonic() - _t0)
del _llm
print(f'Speed: {_speed:.0f} tokens/sec')
if _speed < 2000:
    raise RuntimeError(
        f'Only {_speed:.0f} tok/s — this is CPU speed, not H100.\n'
        'Re-run Cell 6A to rebuild llama-cpp-python with CUDA, then re-run this cell.'
    )
print(f'GPU confirmed ({_speed:.0f} tok/s). Starting benchmark...')

# ── WikiText-2 ───────────────────────────────────────────────────────
wt2 = Path('/tmp/wt2.txt')
if not wt2.exists():
    print('Writing WikiText-2 test split...')
    ds = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
    wt2.write_text('\n\n'.join(ds['text']))
wt2_bytes = wt2.read_bytes()
print(f'WikiText-2: {len(wt2_bytes) // 1024} KB')

# ── Perplexity kernel ────────────────────────────────────────────────
def _log_softmax(x):
    x = x.astype(np.float32)
    x -= x.max(axis=-1, keepdims=True)
    np.exp(x, out=x)
    x /= x.sum(axis=-1, keepdims=True)
    return np.log(x + 1e-30)

def compute_ppl(gguf_path):
    llm = Llama(model_path=str(gguf_path), n_ctx=CONTEXT,
                n_gpu_layers=-1, logits_all=True, verbose=False)
    tokens = llm.tokenize(wt2_bytes, add_bos=True)
    n_tokens = len(tokens)
    print(f'  Tokens: {n_tokens}')

    nlls, prev_end = [], 0
    windows = list(range(0, n_tokens - 1, STRIDE))

    for w, begin in enumerate(windows):
        end   = min(begin + CONTEXT, n_tokens)
        chunk = tokens[begin:end]
        if len(chunk) < 2:
            break

        llm.reset()
        llm.eval(chunk)

        # scores[i] = logits predicting chunk[i+1]
        logits = np.array(llm.scores[:len(chunk)], dtype=np.float32)
        lp     = _log_softmax(logits)

        first = max(0, prev_end - begin)   # skip already-counted context
        for i in range(first, len(chunk) - 1):
            nlls.append(-lp[i, chunk[i + 1]])

        prev_end = end
        if (w + 1) % 50 == 0 or end == n_tokens:
            print(f'  window {w+1}/{len(windows)}  '
                  f'running PPL={math.exp(float(np.mean(nlls))):.4f}')
        if end == n_tokens:
            break

    return math.exp(float(np.mean(nlls))), len(nlls)

# ── Resume from partial results ──────────────────────────────────────
rows = []
if OUTPUT.exists():
    rows = [r for r in json.loads(OUTPUT.read_text())
            if r.get('source') == 'rerun_local']
    if rows:
        print(f'Resuming — already done: {[r["name"] for r in rows]}')

# ── Main loop ────────────────────────────────────────────────────────
for q, info in QUANTS.items():
    row_name = f'llama_cpp_{q}'
    if any(r['name'] == row_name for r in rows):
        print(f'[{q}] already done, skipping')
        continue

    dest = GGUF_DIR / info['file']
    if not dest.exists():
        print(f'Downloading {info["file"]}...')
        hf_hub_download(repo_id=GGUF_REPO, filename=info['file'],
                        local_dir=str(GGUF_DIR))
    print(f"\n{'='*60}\n{q} ({dest.stat().st_size // 1024 // 1024} MB)")

    t0 = time.monotonic()
    ppl, n_ev = compute_ppl(dest)
    wall = time.monotonic() - t0
    print(f'>>> {q}: PPL={ppl:.4f}, n_tokens_eval={n_ev}, {wall:.1f}s')

    rows.append({
        'name':              row_name,
        'description':       f'llama-cpp-python {q}, stride-matched eval.',
        'bpw_reported':      info['bpw'],
        'perplexity':        round(ppl, 4),
        'n_tokens':          n_ev,
        'stride':            STRIDE,
        'context':           CONTEXT,
        'wallclock_seconds': round(wall, 2),
        'model':             'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
        'model_revision':    None,
        'dataset':           'wikitext/wikitext-2-raw-v1/test',
        'source':            'rerun_local',
        'notes':             'llama-cpp-python + TheBloke GGUF. '
                             'stride=512, context=2048.',
    })
    OUTPUT.write_text(json.dumps(rows, indent=2) + '\n')
    print(f'Saved {len(rows)}/4 rows to {OUTPUT}')

print(f'\nDone. {len(rows)}/4 quants. source=rerun_local, stride={STRIDE}')


In [ ]:
# Cell 7: Phase E — generate the perplexity-vs-bpw plot
import os
os.makedirs("docs", exist_ok=True)
!python -m research.quant.plot_results \
    --inputs research/quant/results/srd_sweep.json,research/quant/results/kquant_sweep.json \
    --output docs/srd_perplexity_vs_bpw.png
from IPython.display import Image
Image("docs/srd_perplexity_vs_bpw.png")

In [ ]:
# Cell 8: Show the verdict + download artifacts
import json
from pathlib import Path

srd = json.load(open("research/quant/results/srd_sweep.json"))
kq  = json.load(open("research/quant/results/kquant_sweep.json"))

print("─── SRD sweep ──────────────────────────────────────")
print(f"{'name':<32} {'bpw':>6} {'PPL':>8}")
for r in srd:
    print(f"{r['name']:<32} {r['bpw_reported']:>6.2f} {r['perplexity']:>8.3f}")

print("\n─── K-quant baseline ───────────────────────────────")
print(f"{'name':<32} {'bpw':>6} {'PPL':>8}  source")
for r in kq:
    print(f"{r['name']:<32} {r['bpw_reported']:>6.2f} {r['perplexity']:>8.3f}  {r.get('source', '?')}")

print("\n─── Verdict (pre-committed decision rule) ──────────")
srd_a1 = next((r for r in srd if r['name'].startswith('srd_alpha_1.0_g')), None)
q6 = next((r for r in kq if 'Q6_K' in r['name']), None)
if srd_a1 and q6:
    margin = q6['perplexity'] - srd_a1['perplexity']
    print(f"SRD α=1.0 @ {srd_a1['bpw_reported']:.2f} bpw: PPL = {srd_a1['perplexity']:.3f}")
    print(f"Q6_K     @ {q6['bpw_reported']:.2f} bpw: PPL = {q6['perplexity']:.3f}")
    print(f"Margin (Q6_K - SRD): {margin:+.3f}")
    if margin >= 0.05:
        print("→ VERDICT: SRD beats Q6_K by ≥0.05 PPL. Worth pursuing per the plan.")
    elif margin > 0:
        print("→ VERDICT: SRD edges Q6_K but inside the 0.05 noise margin. Not worth the ~2× memory.")
    else:
        print("→ VERDICT: SRD loses to Q6_K at lower bpw. Shelve.")

# Trigger Colab download — works in Colab only.
try:
    from google.colab import files
    for path in ["research/quant/results/srd_sweep.json",
                 "research/quant/results/kquant_sweep.json",
                 "docs/srd_perplexity_vs_bpw.png"]:
        if Path(path).exists():
            files.download(path)
except ImportError:
    print("\n(google.colab not available — artifacts left at the paths above)")

## Phase D — Mistral-7B Scale-up

Re-runs the SRD sweep on `mistralai/Mistral-7B-v0.1` (~14 GB) to check whether the
TinyLlama-1.1B findings hold at 7B scale.

**Hardware required: A100 (40 GB) or H100.** T4 (16 GB) will OOM loading Mistral-7B in FP16.
Runtime → Change runtime type → A100/H100 before running Cells D1–D4.

Wall-clock: ~20–40 min total (model download + 5 PPL configs on A100).

In [ ]:
# Cell D1: Phase D — Mistral-7B SRD sweep
#
# A100/H100 required — Mistral-7B weights are ~14 GB in FP16.
# Same five configs as TinyLlama: FP16, alpha={0, 0.5, 1.0} per-block g=64,
# and alpha=1.0 per-tensor. Writes srd_sweep_mistral7b.json.
# Wall-clock: ~20-35 min on A100.
!python -m research.quant.bench_perplexity \
    --model mistralai/Mistral-7B-v0.1 \
    --group-size 64 \
    --output research/quant/results/srd_sweep_mistral7b.json


In [ ]:
# Cell D2: Mistral-7B K-quant baseline (cited)
#
# Cites community-reported llama.cpp PPLs for Mistral-7B-v0.1.
# Same stride-mismatch caveat as TinyLlama K-quant rows applies —
# cited numbers use stride=context, ours use stride=512.
!python -m research.quant.bench_llamacpp \
    --model mistralai/Mistral-7B-v0.1 \
    --output research/quant/results/kquant_sweep_mistral7b.json
print('Done — source=published_cite (stride caveat applies)')


In [ ]:
# Cell D3: Mistral-7B scale-up plot
import os
os.makedirs('docs', exist_ok=True)
!python -m research.quant.plot_results \
    --inputs research/quant/results/srd_sweep_mistral7b.json,research/quant/results/kquant_sweep_mistral7b.json \
    --output docs/srd_perplexity_vs_bpw_mistral7b.png
from IPython.display import Image
Image('docs/srd_perplexity_vs_bpw_mistral7b.png')


In [ ]:
# Cell D4: Scale-up verdict + download
import json
from pathlib import Path

try:
    srd_1b = json.load(open('research/quant/results/srd_sweep.json'))
    srd_7b = json.load(open('research/quant/results/srd_sweep_mistral7b.json'))
    kq_7b  = json.load(open('research/quant/results/kquant_sweep_mistral7b.json'))

    def _find(rows, prefix):
        return next((r for r in rows if r['name'].startswith(prefix)), None)

    fp16_1b = _find(srd_1b, 'fp16')
    fp16_7b = _find(srd_7b, 'fp16')
    a0_1b   = _find(srd_1b, 'srd_alpha_0.0')
    a0_7b   = _find(srd_7b, 'srd_alpha_0.0')
    a1_7b   = _find(srd_7b, 'srd_alpha_1.0_g')
    q4_7b   = _find(kq_7b,  'llama_cpp_Q4_K_M')
    q6_7b   = _find(kq_7b,  'llama_cpp_Q6_K')

    print('─── FP16 degradation at α=0 (4.5 bpw) ───────────────────────')
    if a0_1b and fp16_1b:
        d1b = a0_1b['perplexity'] - fp16_1b['perplexity']
        print(f'TinyLlama-1.1B  FP16={fp16_1b["perplexity"]:.4f}  '
              f'α=0={a0_1b["perplexity"]:.4f}  Δ={d1b:+.4f}')
    if a0_7b and fp16_7b:
        d7b = a0_7b['perplexity'] - fp16_7b['perplexity']
        print(f'Mistral-7B-v0.1 FP16={fp16_7b["perplexity"]:.4f}  '
              f'α=0={a0_7b["perplexity"]:.4f}  Δ={d7b:+.4f}')

    print('\n─── SRD α=0 vs Q4_K_M on Mistral-7B ──────────────────────')
    if a0_7b and q4_7b:
        margin = q4_7b['perplexity'] - a0_7b['perplexity']
        print(f'SRD α=0 @ {a0_7b["bpw_reported"]:.2f} bpw : PPL = {a0_7b["perplexity"]:.4f}')
        print(f'Q4_K_M  @ {q4_7b["bpw_reported"]:.2f} bpw : PPL = {q4_7b["perplexity"]:.4f}')
        if margin > 0:
            print(f'→ SRD α=0 BEATS Q4_K_M by {margin:.4f} PPL (holds at 7B scale)')
        else:
            print(f'→ SRD α=0 LOSES to Q4_K_M by {abs(margin):.4f} PPL (does NOT scale)')

    print('\n─── Verdict (pre-committed rule: SRD α=1.0 vs Q6_K) ───────')
    if a1_7b and q6_7b:
        margin6 = q6_7b['perplexity'] - a1_7b['perplexity']
        print(f'SRD α=1.0 @ {a1_7b["bpw_reported"]:.2f} bpw : PPL = {a1_7b["perplexity"]:.4f}')
        print(f'Q6_K      @ {q6_7b["bpw_reported"]:.2f} bpw : PPL = {q6_7b["perplexity"]:.4f}')
        print(f'Margin (Q6_K - SRD α=1.0): {margin6:+.4f}')
        if margin6 >= 0.05:
            print('→ PURSUE — margin ≥0.05 holds at 7B scale')
        elif margin6 > 0:
            print('→ MARGINAL — inside 0.05 noise floor')
        else:
            print('→ SHELVE — SRD loses to Q6_K at 7B')

    try:
        from google.colab import files
        for path in ['research/quant/results/srd_sweep_mistral7b.json',
                     'research/quant/results/kquant_sweep_mistral7b.json',
                     'docs/srd_perplexity_vs_bpw_mistral7b.png']:
            if Path(path).exists():
                files.download(path)
    except ImportError:
        print('\n(google.colab not available — artifacts at paths above)')

except FileNotFoundError as e:
    print(f'Missing: {e}\nRun Cell D1 and Cell D2 first.')
